# Investment Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.8

This notebook solves the investment problem from book section `2.8` with AMPL from Python using `amplpy`.

We solve:
- the original base model with traveler-specific variables
- the aggregated student variation
- the student variation with individual budgets


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```


In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def create_ampl_instance(solver: str = "highs"):
    return ampl_notebook(modules=[solver], license_uuid="default")


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
INVESTMENTS = ["televisions", "sound_systems", "vacuums"]
PEOPLE = ["person_1", "person_2", "person_3", "person_4"]

COST = {
    "televisions": 250000,
    "sound_systems": 180000,
    "vacuums": 70000,
}
SALE = {
    "televisions": 400000,
    "sound_systems": 270000,
    "vacuums": 130000,
}
BUDGETS = {
    "person_1": 3000000,
    "person_2": 4000000,
    "person_3": 2500000,
    "person_4": 1900000,
}


## Problem 1: Base Model

This is the original book model with one variable per investment type and traveler.


In [4]:
@timed
def solve_base_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set I ordered;
        set J ordered;
        param cost {I};
        param sale {I};
        param global_budget >= 0;
        param max_items >= 0;

        var Buy {i in I, j in J} integer >= 0;

        maximize Profit:
            sum {i in I, j in J} (sale[i] - cost[i]) * Buy[i, j];

        subject to GlobalBudget:
            sum {i in I, j in J} cost[i] * Buy[i, j] <= global_budget;

        subject to PersonCap {j in J}:
            sum {i in I} Buy[i, j] <= max_items;
        '''
    )
    ampl.set["I"] = INVESTMENTS
    ampl.set["J"] = PEOPLE
    ampl.param["cost"] = COST
    ampl.param["sale"] = SALE
    ampl.param["global_budget"] = 3000000
    ampl.param["max_items"] = 12
    ampl.solve(solver=solver)

    values = ampl.get_variable("Buy").get_values().to_dict()
    return {
        "positive_buys": {key: int(round(value)) for key, value in values.items() if int(round(value)) > 0},
        "profit": int(round(ampl.get_objective("Profit").value())),
    }


In [5]:
base_result, base_time = solve_base_with_ampl()
print("=== BASE INVESTMENT RESULTS WITH AMPL ===")
print(f"Solution -> {base_result}")
print(f"Time     -> {base_time:.8f} seconds")

assert base_result["profit"] == 2520000


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 2520000
2 simplex iterations
0 branching nodes
=== BASE INVESTMENT RESULTS WITH AMPL ===
Solution -> {'positive_buys': {('vacuums', 'person_1'): 6, ('vacuums', 'person_2'): 12, ('vacuums', 'person_3'): 12, ('vacuums', 'person_4'): 12}, 'profit': 2520000}
Time     -> 1.58596654 seconds


## Problem 2: Aggregated Student Variation

The first variation uses only total purchases by artifact type.


In [6]:
@timed
def solve_aggregated_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set I ordered;
        param cost {I};
        param sale {I};
        param global_budget >= 0;
        param total_item_cap >= 0;

        var Buy {i in I} integer >= 0;

        maximize Profit:
            sum {i in I} (sale[i] - cost[i]) * Buy[i];

        subject to GlobalBudget:
            sum {i in I} cost[i] * Buy[i] <= global_budget;

        subject to TotalItemCap:
            sum {i in I} Buy[i] <= total_item_cap;
        '''
    )
    ampl.set["I"] = INVESTMENTS
    ampl.param["cost"] = COST
    ampl.param["sale"] = SALE
    ampl.param["global_budget"] = 3000000
    ampl.param["total_item_cap"] = 48
    ampl.solve(solver=solver)

    values = ampl.get_variable("Buy").get_values().to_dict()
    return {
        "purchases": {key: int(round(value)) for key, value in values.items()},
        "profit": int(round(ampl.get_objective("Profit").value())),
    }


In [7]:
aggregated_result, aggregated_time = solve_aggregated_with_ampl()
print("=== AGGREGATED VARIATION RESULTS WITH AMPL ===")
print(f"Solution -> {aggregated_result}")
print(f"Time     -> {aggregated_time:.8f} seconds")

assert aggregated_result["profit"] == 2520000


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 2520000
1 simplex iterations
1 branching nodes
=== AGGREGATED VARIATION RESULTS WITH AMPL ===
Solution -> {'purchases': {'televisions': 0, 'sound_systems': 0, 'vacuums': 42}, 'profit': 2520000}
Time     -> 1.57681192 seconds


## Problem 3: Student Variation with Individual Budgets

The second variation gives each traveler a different budget.


In [8]:
@timed
def solve_individual_budget_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set I ordered;
        set J ordered;
        param cost {I};
        param sale {I};
        param budget {J} >= 0;
        param max_items >= 0;

        var Buy {i in I, j in J} integer >= 0;

        maximize Profit:
            sum {i in I, j in J} (sale[i] - cost[i]) * Buy[i, j];

        subject to PersonBudget {j in J}:
            sum {i in I} cost[i] * Buy[i, j] <= budget[j];

        subject to PersonCap {j in J}:
            sum {i in I} Buy[i, j] <= max_items;
        '''
    )
    ampl.set["I"] = INVESTMENTS
    ampl.set["J"] = PEOPLE
    ampl.param["cost"] = COST
    ampl.param["sale"] = SALE
    ampl.param["budget"] = BUDGETS
    ampl.param["max_items"] = 12
    ampl.solve(solver=solver)

    values = ampl.get_variable("Buy").get_values().to_dict()
    return {
        "positive_buys": {key: int(round(value)) for key, value in values.items() if int(round(value)) > 0},
        "profit": int(round(ampl.get_objective("Profit").value())),
    }


In [9]:
individual_budget_result, individual_budget_time = solve_individual_budget_with_ampl()
print("=== INDIVIDUAL-BUDGET VARIATION RESULTS WITH AMPL ===")
print(f"Solution -> {individual_budget_result}")
print(f"Time     -> {individual_budget_time:.8f} seconds")

assert individual_budget_result["profit"] == 6330000


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 6330000
12 simplex iterations
1 branching nodes
=== INDIVIDUAL-BUDGET VARIATION RESULTS WITH AMPL ===
Solution -> {'positive_buys': {('televisions', 'person_1'): 12, ('televisions', 'person_2'): 12, ('televisions', 'person_3'): 9, ('televisions', 'person_4'): 6, ('vacuums', 'person_3'): 3, ('vacuums', 'person_4'): 5}, 'profit': 6330000}
Time     -> 1.51269375 seconds
